# Persona Engine — Stage 7A: Controlled Base Model Selection Experiment
## Fine-Tuning Qwen/Qwen2.5-3B-Instruct with QLoRA

**Objective:** Determine whether moving from 1.5B (`Qwen/Qwen2.5-1.5B-Instruct`) to ~3B (`Qwen/Qwen2.5-3B-Instruct`) produces a meaningfully better Persona Engine under strictly controlled conditions.

### Experimental Controls
- **Base Model:** `Qwen/Qwen2.5-3B-Instruct` (vs Control `Qwen/Qwen2.5-1.5B-Instruct`)
- **Quantization:** 4-bit NF4 (`load_in_4bit=True`, `compute_dtype=float16`)
- **LoRA Config:** $r=16, \alpha=32, \text{dropout}=0.05$, target modules: `[q_proj, k_proj, v_proj, o_proj]`
- **Loss Masking:** Assistant-only masking (System & User tokens masked to -100)
- **Hyperparameters:** $\text{lr}=2\times 10^{-4}$, $\text{batch}=2$, $\text{accum}=4$ (effective batch 8), $\text{epochs}=3$, cosine schedule, warmup 5%
- **Seed:** 42
- **Dataset:** Frozen Stage 5 splits (Train=3,760, Val=868, Test=1,310)
- **Adapter Output:** `ml/models/adapters/stage7a_qwen2.5_3b`

## 1. Hardware & GPU Diagnostics

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
    print('Total VRAM (GB) :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    raise SystemError('GPU required! Please select Runtime -> Change runtime type -> T4 or A100 GPU.')

## 2. Environment Setup & Dependency Installation

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets trl
print('Dependencies installed successfully.')

## 3. Clone Repository or Mount Google Drive

In [ ]:
# If running directly from git repository:
# !git clone https://github.com/Vivek-1109/persona-engine.git
# %cd persona-engine

# Or verify files locally
import os
print('Working Directory:', os.getcwd())

## 4. Pre-Flight Dataset Fingerprint Audit & Sanity Check

In [ ]:
!python ml/scripts/run_stage7a_experiment.py --dry-run --sanity-check

## 5. Execute Controlled Stage 7A Training

In [ ]:
!python ml/scripts/run_stage7a_experiment.py \
    --base-model Qwen/Qwen2.5-3B-Instruct \
    --adapter-output ml/models/adapters/stage7a_qwen2.5_3b \
    --seed 42 \
    --epochs 3 \
    --batch-size 2 \
    --grad-accum 4 \
    --lr 2e-4

## 6. Evaluate and Compare with Baseline

In [ ]:
!python ml/scripts/run_stage7a_evaluation.py